# Exp 21: Face erase on non-VIP images

Card: [exp21_face_erase.md](../../experiments/exp21_face_erase.md)

In [ ]:
import os

import matplotlib.pyplot as plt
import mlflow
import numpy as np
from matplotlib.patches import Rectangle
from PIL import Image
from sklearn.metrics import average_precision_score, precision_recall_curve

from core.boxes import FACE_SOURCE, FACE_THRESHOLD, pseudo_boxes
from core.config import CLASSES, MLFLOW_URI, ROOT, TRAIN_DIR
from core.cv import VIP, Config, CrossValidator, CVResult
from core.data import FACE_BOXES, read_boxes, read_train
from core.gradcam import GradCamFigures, compare

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-face-erase-smoke",
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
    N_BOXES = 2  # Cat, Dog images per face box figure
    N_SAMPLE = 2  # Cat, Dog images per comparison figure
    N_POSITIVES = 2  # images per VIP Grad-CAM figure
else:
    OVERRIDES = {}
    N_BOXES = 16
    N_SAMPLE = 16  # exp19 sample
    N_POSITIVES = None  # all labeled positives
SEED = 0  # exp19 sample
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIG = Config.from_toml(
    CONFIG_DIR / "exp21_face_erase" / "resnet18.toml", **OVERRIDES
)
EXP16 = Config.from_toml(CONFIG_DIR / "exp16_downsampling" / "resnet18.toml")
EXP20 = Config.from_toml(FACE_SOURCE)
FOLD = 0
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp21_face_erase"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode
SERIES = {"exp16": "#eb6834", "exp20": "#2a78d6", "face erase": "#1baf7a"}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)

mlflow.set_tracking_uri(MLFLOW_URI)
client = mlflow.MlflowClient()

## Face boxes

In [ ]:
if not FACE_BOXES.exists():  # same as `python -m core.boxes --face`
    pseudo_boxes(read_train(), EXP20, FACE_THRESHOLD, classes=("Cat", "Dog")).to_csv(FACE_BOXES, index=False)
boxes = read_boxes(FACE_BOXES)
train = read_train()
train = train[(train["fold"] != FOLD) & ~train[VIP].any(axis=1)]  # erased rows: fold 0 train part, non-VIP
figures = {}
fig, axes = plt.subplots(4, N_BOXES // 2, figsize=(1.1 * N_BOXES, 9), squeeze=False)
for k, cls in enumerate(["Cat", "Dog"]):
    ids = train.loc[train[cls] == 1, "image_id"].sample(N_BOXES, random_state=SEED)
    for ax, image_id in zip(axes[2 * k : 2 * k + 2].flat, ids):
        ax.imshow(Image.open(TRAIN_DIR / f"{image_id}.jpg").convert("RGB").resize((224, 224)))
        for c, (x0, y0, x1, y1) in boxes.get(image_id, []):
            if c == cls:
                ax.add_patch(Rectangle((x0 * 224, y0 * 224), (x1 - x0) * 224, (y1 - y0) * 224, fill=False, color="#eb6834", lw=2))
        ax.set_title(cls, fontsize=8)
for ax in axes.flat:
    ax.set_axis_off()
fig.suptitle(f"Face boxes · Grad-CAM >= {FACE_THRESHOLD} of max, exp20 · fold {FOLD} train part, non-VIP", color="#52514e")
fig.tight_layout()
figures[f"{PREFIX}_face_boxes.png"] = fig
plt.show()

## Training

In [ ]:
results = {"exp16": CVResult.load(EXP16), "exp20": CVResult.load(EXP20)}
results["face erase"] = CrossValidator(CONFIG).run()

## Grad-CAM Cat, Dog: exp16, exp20, face erase

In [ ]:
cams = {"exp16": GradCamFigures(EXP16), "exp20": GradCamFigures(EXP20), "face erase": GradCamFigures(CONFIG)}
source = "face erase" if SMOKE else "exp16"  # exp19 rows drawn from exp16 OOF; smoke OOF holds few rows
for cls in ["Cat", "Dog"]:
    rows = cams[source].positive_rows(cls, N_SAMPLE, SEED)
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_compare.png"] = compare(cams, cls, rows)
plt.show()

## Grad-CAM VIP: unseen vs trained images

In [ ]:
for cls in VIP:
    for train_view in (False, True):
        fig = cams["face erase"].positives(cls, train=train_view, n=N_POSITIVES)
        split = "train" if train_view else "validation"
        figures[f"{PREFIX}_grad_cam_{cls.lower()}_{split}.png"] = fig
plt.show()

## Precision-recall curves

In [ ]:
fig, axes = plt.subplots(
    1, len(CLASSES), figsize=(3.4 * len(CLASSES), 3.6), sharey=True
)
for cls, ax in zip(CLASSES, axes):
    for name, r in results.items():
        y, p = r.oof[cls], r.oof[f"p_{cls}"]
        precision, recall, _ = precision_recall_curve(y, p)
        ax.plot(
            recall,
            precision,
            color=SERIES[name],
            lw=2,
            drawstyle="steps-post",
            label=f"{name}, AP {average_precision_score(y, p):.3f}",
        )
    ax.set(
        title=f"{cls} ({int(y.sum())} positives)",
        xlabel="recall",
        xlim=(0, 1.02),
        ylim=(0, 1.05),
    )
    ax.legend(loc="lower left", alignment="left")
axes[0].set_ylabel("precision")
fig.tight_layout()
figures[f"{PREFIX}_pr_curves.png"] = fig
plt.show()

In [ ]:
with mlflow.start_run(
    experiment_id=client.get_experiment_by_name(CONFIG.experiment).experiment_id,
    run_name="summary",
):
    mlflow.log_param("val_mode", "fast")
    if SAVE_FIGURES:
        for fname, fig in figures.items():
            fig.savefig(FIGURES / fname, dpi=120)
            mlflow.log_figure(fig, fname)